In [33]:
import json
import os
import time
from pathlib import Path
from sentence_transformers import SentenceTransformer, util
import numpy as np
from openai import OpenAI


model = SentenceTransformer('multi-qa-MiniLM-L6-cos-v1')

API_KEY_PATH = Path('openai_key.txt')



In [2]:
def rag_based_relevance(x: str, ad_texts: list[str]):
    x_embedding = model.encode(x)
    ads_embedding = model.encode(ad_texts)
    bf = (1 + util.dot_score(x_embedding, ads_embedding)[0].numpy()) / 2
    return bf

def f_hat(q, eta=1.0, beta=1.0):
    """f_hat(q) = f(q) / C."""
    return eta * (q ** beta)


In [3]:
def load_openai_api_key():
    env_key = os.environ.get('OPENAI_API_KEY')
    if env_key:
        return env_key.strip()

    if API_KEY_PATH.exists():
        file_key = API_KEY_PATH.read_text().strip()
        if file_key:
            return file_key

    raise EnvironmentError('Set OPENAI_API_KEY or create openai_key.txt in the notebook directory.')



In [4]:
def run_auction(context, bids, advertisers, ads, q0,
                eta=1.0, beta=1.0, lambda_tilde=1.0):
    b     = np.array([bids[name] for name in advertisers])
    q_raw = np.array([float(v) for v in rag_based_relevance(context, ads)])

    f0             = f_hat(q0, eta, beta)
    reserve_prices = f0 / q_raw
    eligible_mask  = b >= reserve_prices
    eligible_idx   = np.where(eligible_mask)[0]

    q_eligible     = q_raw[eligible_idx]
    b_eligible     = b[eligible_idx]
    names_eligible = [advertisers[i] for i in eligible_idx]
    m              = len(eligible_idx)

    q_full     = np.concatenate([[q0], q_eligible])
    q_tilde    = q_full / q_full.sum()
    f_tilde_q0 = f_hat(q0, eta, beta) / q_full.sum()

    s       = np.concatenate([[f_tilde_q0], q_tilde[1:] * b_eligible])
    log_num = np.log(q_tilde) + s / lambda_tilde
    log_num -= log_num.max()
    weights = np.exp(log_num)
    x       = weights / weights.sum()

    def compute_payment(i_local):
        qt_i = q_tilde[1 + i_local]
        bi   = b_eligible[i_local]
        ri   = reserve_prices[eligible_idx[i_local]]
        xi_b = x[1 + i_local]

        b_cf = b_eligible.astype(float).copy()
        b_cf[i_local] = ri
        s_cf   = np.concatenate([[f_tilde_q0], q_tilde[1:] * b_cf])
        log_cf = np.log(q_tilde) + s_cf / lambda_tilde
        log_cf -= log_cf.max()
        x_cf   = np.exp(log_cf) / np.exp(log_cf).sum()
        xi_ri  = x_cf[1 + i_local]

        payment = bi * (xi_b - 1) + ri + (lambda_tilde / qt_i) * np.log(xi_b / xi_ri)
        return payment, xi_ri

    payments, xi_ri_list = [], []
    for i_local in range(m):
        p, xi_ri = compute_payment(i_local)
        payments.append(p)
        xi_ri_list.append(xi_ri)

    names_full = ["organic"] + names_eligible
    summary = {
        "q_0": q0,
        "q_raw": q_raw, 
        "reserve_prices": reserve_prices, 
        "eligible_mask": eligible_mask, 
        "q_tilde": q_tilde, 
        "names_eligible": names_eligible, 
        "x": x, 
        "payments": payments, 
        "xi_ri_list": xi_ri_list, 
        "b_eligible": b_eligible,
        "eligible_idx": eligible_idx, 
        "f_tilde_q0": f_tilde_q0, 
        "q_full_sum": q_full.sum() 
    }
    return x, names_full, payments, eligible_idx, summary



def generate_sentence(original_query, ad_text, advertiser_name,
                      previous_queries, previous_outputs,
                      organic_doc=None,
                      model_name="gpt-4-turbo", temperature=1.0):
    client = OpenAI(api_key=load_openai_api_key())

    if len(previous_outputs) == 0:
        if ad_text is None:
            query = f'''{original_query}
                    Please respond to this question in only one sentence.
                    You may use the following background context to inform your answer:
                    >> {organic_doc}'''
        else:
            query = f'''{original_query}
                    Please respond to this question in only one sentence while also advertise {advertiser_name} with this context >>
                    {ad_text}
                    Make sure to connect the answer and the advertisement very naturally,
                    not something like appending the ads after just answering the question.
                    Focus on answering the question, there shouldn't be too much advertisement in the output.
                    Make sure the response is one sentence.'''
    else:
        if ad_text is None:
            query = f'''You must continue your answer to my original query: {original_query} 
                    Your previous response was >> {previous_outputs[-1]}
                    Please add exactly one new sentence at the end that continues naturally without modifying, paraphrasing, or removing any part of your previous response.
                    You may use the following content to inform the new sentence:
                    >> {organic_doc}
                    Do NOT mention or reference any advertisement in the new sentence.
                    Make sure there is exactly one new sentence added.
                    Write the entire document, which merges your previous response and the new sentence.'''
        else:
            query = f'''You must continue your answer to my original query: {original_query} 
                    Your previous response was >> {previous_outputs[-1]}
                    And you now should advertise {advertiser_name}, but without hurting the coherency of the entire document.
                    Here's some context about {advertiser_name}:
                    >> {ad_text}
                    Make sure there is exactly one new sentence added.
                    Write the entire document, which merges your previous response and the new sentence.'''

    messages = [{"role": "user", "content": query}]

    response = client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        max_tokens=300,
        messages=messages,
    )
    return query, response.choices[0].message.content.strip()

In [ ]:
def generate_organic_doc(original_query, previous_outputs,
                         model_name="gpt-4-turbo", temperature=1.0):

    client = OpenAI(api_key=load_openai_api_key())

    if len(previous_outputs) == 0:
        query = f'''For the following question, write a summarized answer 
                of 3 sentences that provides factual, neutral context relevant to the topic.
                Write like a Wikipedia excerpt.
                Question: {original_query}'''
    else:
        query = f'''For the following question, write a summarized answer
                of 3 sentences that introduces new relevant information not yet mentioned in the partial answer,
                providing factual, neutral context to help continue the answer.
                Do NOT repeat or comment on anything already mentioned in the partial answer.
                Write like a Wikipedia excerpt.
                Question: {original_query}
                Partial answer so far: {previous_outputs[-1]}'''

    messages = [{"role": "user", "content": query}]
    # organic response for q0
    response = client.chat.completions.create(
        model=model_name,
        temperature=temperature,
        max_tokens=500,
        messages=messages,
    )
    return response.choices[0].message.content.strip()

In [6]:
def run_sequential_auction(
    bids_dict,
    advertisers,  
    ads,          
    original_query,
    eta_val       = 1.0,
    beta_val      = 1.0,
    lambda_val    = 1.0,
    with_replacement = False,
    seed          = 42,
    n_rounds      = 3,
):
    rng = np.random.default_rng(seed=seed)
    previous_queries = []
    previous_outputs = []
    round_logs       = []
    used_advertisers = set()

    print("=" * 70)
    print(f"SEQUENTIAL {n_rounds}-SENTENCE GENERATION WITH AD AUCTION")
    print(f"  replacement={with_replacement}  eta={eta_val}  beta={beta_val}  lambda={lambda_val}")
    print("=" * 70)
    print(f"Original query: {original_query}\n")

    for t in range(1, n_rounds + 1):
        print(f"{'─' * 70}")
        print(f"ROUND {t}")

        auction_context = original_query if len(previous_outputs) == 0 \
                          else original_query + " " + previous_outputs[-1]
        print(f"Context fed to auction: {auction_context[:120]}...")

        # Step 0: Generate organic background doc, compute q0
        print("  → Generating organic background document...")
        organic_doc = generate_organic_doc(          
            original_query  = original_query,
            previous_outputs = previous_outputs,
        )
        q0_val = float(rag_based_relevance(auction_context, [organic_doc])[0])  
        print(f"  → Organic doc          : \"{organic_doc}\"")


        # Step 1: Build effective bids
        if with_replacement:
            effective_bids = bids_dict.copy()
        else:
            effective_bids = {
                name: (0 if name in used_advertisers else bid)
                for name, bid in bids_dict.items()
            }
            if used_advertisers:
                print(f"  → Zeroed-out bids for used ads: {used_advertisers}")

        # Step 2: Run auction
        x, names_full, payments, eligible_idx, summ = run_auction(
            context      = auction_context,
            bids         = effective_bids,
            advertisers  = advertisers,
            ads          = ads,
            q0           = q0_val,
            eta          = eta_val,
            beta         = beta_val,
            lambda_tilde = lambda_val,
        )
        
        
        print(f"  → q0                   : {float(summ['q_0']):.4f}")
        # Print auction result with bid and reserve price
        print("\nAuction result:")
        print(f"  {'Name':22s}  {'bid':>5}  {'reserve':>8}  {'q̃_i':>7}  {'eligible':>8}  {'x_i':>7}")
        
        # organic row
        print(f"  {'organic':22s}  {'—':>5}  {'—':>8}  {summ['q_tilde'][0]:>7.4f}  {'—':>8}  {x[0]:.4f}")
        
        # advertiser rows
        for i, name in enumerate(advertisers):
            bid_val = effective_bids[name]
            res     = summ["reserve_prices"][i]
            elig    = "✓" if summ["eligible_mask"][i] else "✗"
            used_tag = " (used)" if name in used_advertisers else ""

            # find position in names_full to get q_tilde and x
            if name in names_full:
                pos     = names_full.index(name)
                qt_str  = f"{summ['q_tilde'][pos]:>7.4f}"
                x_str   = f"{x[pos]:.4f}"
            else:
                qt_str  = f"{'—':>7}"
                x_str   = "—"

            print(f"  {name+used_tag:22s}  {bid_val:>5}  {res:>8.4f}  {qt_str}  {elig:>8}  {x_str:>7}")

        if payments:
            print("  Per-click payments:")
            for name, p in zip(summ["names_eligible"], payments):
                print(f"    {name:15s}  p̃ = {p:.4f}")

        # Step 3: Sample winner
        winner_idx  = int(rng.choice(len(x), p=x))
        winner_name = names_full[winner_idx]
        print(f"\n  → Sampled winner: [{winner_name}]  (x = {x[winner_idx]:.4f})")

        # Step 4: Generate output
        if winner_idx == 0:
            print("  → Generating ORGANIC sentence using background doc as context")
            query_used, new_output = generate_sentence(
                original_query   = original_query,
                ad_text          = None,
                advertiser_name  = None,
                previous_queries = previous_queries,
                previous_outputs = previous_outputs,
                organic_doc      = organic_doc,      
            )
        else:
            ad_idx         = eligible_idx[winner_idx - 1]
            ad_text_gen    = ads[ad_idx]
            advertiser_gen = advertisers[ad_idx]
            if not with_replacement:
                used_advertisers.add(winner_name)
            print(f"  → Regenerating sentence WITH ad from [{winner_name}]")
            query_used, new_output = generate_sentence(
                original_query   = original_query,
                ad_text          = ad_text_gen,
                advertiser_name  = advertiser_gen,
                previous_queries = previous_queries,
                previous_outputs = previous_outputs,
            )

        previous_queries.append(query_used)
        previous_outputs.append(new_output)

        print(f"\n  Full output after round {t}:")
        print(f'  "{new_output}"')

        round_logs.append({
            "round":       t, 
            "winner":      winner_name, 
            "ad_injected": winner_idx != 0, 
            "q0":          float(summ["q_0"]), 
            "winner_q_raw": float(summ["q_raw"][summ["eligible_idx"][winner_idx - 1]]) if winner_idx != 0 else None,
            "allocation":  dict(zip(names_full, x.tolist())),
            "winner_x": x[winner_idx], 
            "payments_dict": dict(zip(summ["names_eligible"], payments)), 
            "q_tilde": dict(zip(names_full, summ['q_tilde'].tolist())), 
            "f_tilde_q0": summ["f_tilde_q0"], 
            "q_full_sum": summ["q_full_sum"], 
            "output":      new_output 
        })

    print("\n" + "=" * 70)
    print("FINAL RESPONSE (last output = complete document)")
    print("=" * 70)
    print(previous_outputs[-1])
    print("\nRound-by-round ad log:")
    for log in round_logs:
        tag = f"[AD: {log['winner']}]" if log["ad_injected"] else "[organic]"
        print(f"  Round {log['round']}: {tag}  (q0={log['q0']:.4f})")
        
     # ── Ad injection summary ──────────────────────────────────────────────────
    ad_rounds    = [log for log in round_logs if log["ad_injected"]]
    organic_rounds = [log for log in round_logs if not log["ad_injected"]]
    print(f"\n  Total rounds    : {n_rounds}")
    print(f"  Ad sentences    : {len(ad_rounds)}  {[log['winner'] for log in ad_rounds]}")
    print(f"  Organic sentences: {len(organic_rounds)}")

    return previous_outputs, round_logs



In [7]:


def run_N_trials(
    N,
    bids_dict,
    advertisers,
    ads,
    original_query,
    eta_val, beta_val, lambda_val,
    with_replacement=True,
    n_rounds=3,
    save_path='trial_logs.json',
):
    save_file = Path(save_path)
    
    # 1. Load existing logs if the file exists
    if save_file.exists():
        try:
            with open(save_file, 'r', encoding='utf-8') as f:
                all_logs = json.load(f)
            print(f" Found {len(all_logs)} existing logs in {save_path}.")
        except json.JSONDecodeError:
            print(f" Warning: {save_path} is corrupted. Starting with a fresh list.")
            all_logs = []
    else:
        all_logs = []

    initial_count = len(all_logs)
    
    # 2. Append N new trials to the existing list
    print(f" Starting to append {N} new trials...")

    for i in range(N):
        # We use (initial_count + i) as the seed to ensure that 
        # new trials have unique random sequences different from previous ones.
        current_trial_idx = initial_count + i
        
        print(f"\n{'='*70}")
        print(f"APPENDING TRIAL {i+1}/{N} (Total Sequence: {current_trial_idx + 1})")
        print(f"{'='*70}")
        
        _, logs = run_sequential_auction(
            bids_dict        = bids_dict,
            advertisers      = advertisers,
            ads              = ads,
            original_query   = original_query,
            eta_val          = eta_val,
            beta_val         = beta_val,
            lambda_val       = lambda_val,
            with_replacement = with_replacement,
            seed             = current_trial_idx, 
            n_rounds         = n_rounds,
        )
        
        all_logs.append(logs)
        
        # 3. Save to file immediately after each trial to ensure data safety
        save_file.write_text(json.dumps(all_logs, indent=2, ensure_ascii=False))

    # 4. Final Output: Show the total number of logs now available
    final_total = len(all_logs)
    print(f"\n Done! Added {N} trials.")
    print(f" Total logs now stored in {save_path}: {final_total}")
    
    return all_logs

# Running Experiments

## Scenario 1

In [13]:
prompt = '"What can I visit on a trip to Hawaii?"'

advertisers = ['SunWing Airlines', 'TropicStay', 'WanderBite', 'NovaSkin', 'GridPower Bank']
ads = [
    "Take off with SunWing Airlines, your trusted partner for seamless travel across the Pacific and beyond. With spacious cabins, award-winning in-flight service, and direct routes to the world's most breathtaking destinations, SunWing makes every journey as memorable as the destination itself. Book today and enjoy exclusive deals, flexible cancellation, and a loyalty rewards program that turns miles into memories. Your next adventure starts here — fly with SunWing.",
    "Discover the perfect retreat with TropicStay, the premier platform for booking vacation rentals in the world's most beautiful destinations. From oceanfront villas to cozy jungle hideaways, TropicStay connects travelers with handpicked, locally-owned properties that offer authentic experiences you won't find anywhere else. With instant booking, 24/7 guest support, and verified reviews from real travelers, planning your dream getaway has never been easier. Explore, book, and belong — only with TropicStay.",
    "Unlock the culinary soul of every destination with WanderBite, the app that connects food lovers to the best local restaurants, street food spots, and hidden dining gems around the globe. Whether you're craving fresh poke bowls on the North Shore or a Michelin-starred tasting menu in the city, WanderBite guides you there with curated recommendations, real-time reservations, and exclusive foodie deals. Because the best travel memories are made at the table. Download WanderBite and taste the world.",
    "Introducing NovaSkin, the dermatologist-approved skincare line engineered for the modern lifestyle. From SPF-50 daily moisturizers to overnight repair serums, NovaSkin's lightweight, reef-safe formulas protect and restore your skin whether you're under the office lights or the open sun. Trusted by over 10 million customers worldwide, NovaSkin combines cutting-edge biotechnology with clean, sustainable ingredients to deliver visible results you can feel confident about. Because great skin doesn't take a vacation — and neither does NovaSkin.",
    "Stay charged through every adventure with GridPower Bank, the ultra-slim, high-capacity portable charger built for life on the move. Featuring rapid-charge technology, dual USB-C ports, and a rugged waterproof design, GridPower Bank keeps your devices powered through long flights, beach days, and everything in between. Compact enough to fit in your pocket, powerful enough to charge your laptop twice over — GridPower Bank is the travel essential you didn't know you needed. Power up and go.",
]

bids_dict = {"SunWing Airlines": 3, "TropicStay": 3, "WanderBite": 2, "NovaSkin": 2, "GridPower Bank": 1}

In [14]:
q = rag_based_relevance(prompt, ads)
print([round(float(v), 2) for v in q])
for name, value in zip(advertisers, q):
    print(f'{name}: {value:.4f}')

[0.62, 0.67, 0.61, 0.49, 0.59]
SunWing Airlines: 0.6236
TropicStay: 0.6702
WanderBite: 0.6114
NovaSkin: 0.4874
GridPower Bank: 0.5901


In [18]:
def run_experiment_1(N, eta_val, beta_val, with_replacement, lambda_val=1.0):
    repl_str  = "with" if with_replacement else "without"
    save_path = f'QP_single_{repl_str}_replacement_logs_scenario_1.json'
    print(f"Running: N={N}, eta={eta_val}, beta={beta_val}, {repl_str}_replacement → {save_path}")
    run_N_trials(
        N                = N,
        bids_dict        = bids_dict,
        advertisers      = advertisers,
        ads              = ads,
        original_query   = prompt,
        eta_val          = eta_val,
        beta_val         = beta_val,
        lambda_val       = lambda_val,
        with_replacement = with_replacement,
        save_path        = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path

### with replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_1(N=20, eta_val=2, beta_val=0.8,with_replacement=True)
    except Exception as e:
        print(f"Error: {e}, retrying...")

### without replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_1(N=20, eta_val=2, beta_val=0.8,with_replacement=False)
    except Exception as e:
        print(f"Error: {e}, retrying...")

## Scenario 2

In [8]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
]

bids_dict = {"Velora": 3, "BookHaven": 3, "MassMart": 2, "EspressoEdge": 2}

In [9]:
q = rag_based_relevance(prompt, ads)
print([round(float(v), 2) for v in q])
for name, value in zip(advertisers, q):
    print(f'{name}: {value:.4f}')

[0.53, 0.7, 0.51, 0.49]
Velora: 0.5251
BookHaven: 0.7048
MassMart: 0.5078
EspressoEdge: 0.4913


In [10]:
def run_experiment_2(N, eta_val, beta_val, with_replacement, lambda_val=1.0):
    repl_str  = "with" if with_replacement else "without"
    save_path = f'QP_single_{repl_str}_replacement_logs_scenario_2.json'
    print(f"Running: N={N}, eta={eta_val}, beta={beta_val}, {repl_str}_replacement → {save_path}")
    run_N_trials(
        N                = N,
        bids_dict        = bids_dict,
        advertisers      = advertisers,
        ads              = ads,
        original_query   = prompt,
        eta_val          = eta_val,
        beta_val         = beta_val,
        lambda_val       = lambda_val,
        with_replacement = with_replacement,
        save_path        = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path

### with replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_2(N=20, eta_val=2, beta_val=0.8,with_replacement=True)
    except Exception as e:
        print(f"Error: {e}, retrying...")

### without replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_2(N=20, eta_val=2, beta_val=0.8,with_replacement=False)
    except Exception as e:
        print(f"Error: {e}, retrying...")

## Scenario 3

In [20]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
]

bids_dict = {"Velora": 2, "BookHaven": 1, "MassMart": 3, "EspressoEdge": 3}

In [21]:
q = rag_based_relevance(prompt, ads)
print([round(float(v), 2) for v in q])
for name, value in zip(advertisers, q):
    print(f'{name}: {value:.4f}')

[0.53, 0.7, 0.51, 0.49]
Velora: 0.5251
BookHaven: 0.7048
MassMart: 0.5078
EspressoEdge: 0.4913


In [22]:
def run_experiment_3(N, eta_val, beta_val, with_replacement, lambda_val=1.0):
    repl_str  = "with" if with_replacement else "without"
    save_path = f'QP_single_{repl_str}_replacement_logs_scenario_3.json'
    print(f"Running: N={N}, eta={eta_val}, beta={beta_val}, {repl_str}_replacement → {save_path}")
    run_N_trials(
        N                = N,
        bids_dict        = bids_dict,
        advertisers      = advertisers,
        ads              = ads,
        original_query   = prompt,
        eta_val          = eta_val,
        beta_val         = beta_val,
        lambda_val       = lambda_val,
        with_replacement = with_replacement,
        save_path        = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path

### with replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_3(N=20, eta_val=1.5, beta_val=0.8,with_replacement=True)
    except Exception as e:
        print(f"Error: {e}, retrying...")

### without replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_3(N=20, eta_val=1.5, beta_val=0.8,with_replacement=False)
    except Exception as e:
        print(f"Error: {e}, retrying...")

## Scenario 4

In [27]:
prompt = '"Can you suggest some books similar to \"To Kill a Mockingbird\"?"'

advertisers = ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']
ads = [
    "Discover the future of technology with Velora, the brand that redefines innovation and elegance. Velora designs and sells a premium range of smartphones, tablets, laptops, and smartwatches, all crafted to seamlessly integrate into your lifestyle. Our products are engineered with user-friendly interfaces, stunning designs, and cutting-edge technology to keep you connected and productive. Velora's ecosystem offers unparalleled synchronization across devices, ensuring a smooth and efficient experience whether you're at work, school, or on the go. With Velora Pay, you can enjoy secure and convenient payment services, while our robust cloud service keeps your data safe and accessible anytime, anywhere. Elevate your tech experience with Velora, where sophistication meets simplicity and advanced functionality.",
    "Introducing BookHaven, your ultimate online bookstore where the world of literature is just a click away. At BookHaven, we offer an extensive collection of books spanning every genre and interest, from timeless classics and gripping thrillers to insightful non-fiction and enchanting children's stories. Our user-friendly platform ensures a seamless shopping experience, with personalized recommendations and unbeatable prices. Whether you're a voracious reader or just looking for your next great read, BookHaven is dedicated to delivering literary treasures right to your doorstep with fast, reliable shipping and a hassle-free return policy. Discover the joy of reading with BookHaven, where every book finds its perfect reader. Dive into a world of endless possibilities and let your next adventure begin at BookHaven!",
    "Experience the joy of shopping at MassMart, where quality meets value in a dynamic retail environment tailored for your satisfaction. At MassMart, members enjoy exclusive access to a vast selection of premium, bulk-sized products, from fresh groceries to high-tech electronics, all at unbeatably low prices. With a commitment to customer happiness, sustainability, and community support, MassMart isn't just a shopping destination — it's a part of your community. Dive into a world of savings and discover why millions choose MassMart as their trusted shopping partner. Join us today and see the difference MassMart can make in your shopping experience, where every visit is more than just shopping — it's an adventure!",
    "Experience the warmth and delight of EspressoEdge, where every sip offers an invitation to a world of exquisite flavors and aromas. Renowned globally for its high-quality, handcrafted beverages, EspressoEdge is committed to sourcing the finest Arabica beans, expertly blending them into a variety of rich espressos, frothy cappuccinos, and creamy lattes. Each visit to an EspressoEdge store is more than just a coffee run—it's an opportunity to savor a moment of luxury amid the hustle of daily life. Whether you seek the comfort of a familiar classic or the thrill of a new seasonal specialty, EspressoEdge welcomes all to gather, connect, and enjoy a cup perfectly tailored to your taste. Step into your local EspressoEdge today and join us in celebrating the art of coffee.",
    "Discover the power of connection with SocialHub, the world's leading social media platform. With over two billion active users, SocialHub is your gateway to staying in touch with friends and family, discovering new communities, and sharing your life's moments. Our innovative features, from personalized news feeds to interactive groups, make it easy to engage with what matters most to you. Whether you're promoting your business, staying updated on the latest news, or simply keeping up with loved ones, SocialHub is the ultimate tool to enhance your digital experience. Join us today and be part of a global network where connections come to life!",
    "Experience the refreshing taste of ColaBubbles, the world's favorite soft drink. With its unique blend of flavors and effervescent bubbles, ColaBubbles has been bringing joy to people of all ages for over a century. Whether you're enjoying a moment of relaxation, celebrating with friends, or on the go, ColaBubbles is the perfect companion to quench your thirst and uplift your spirits. Our commitment to quality and tradition ensures every sip is as delightful as the first. Indulge in the classic taste of ColaBubbles and make every moment special. Taste the feeling!",
    "Unleash the bold taste of FizzyPop, the iconic soft drink that invigorates and refreshes like no other. Known for its crisp, refreshing flavor and vibrant effervescence, FizzyPop is the perfect choice for those who dare to live life to the fullest. Whether you're at a party, watching a game, or simply taking a break, FizzyPop brings a burst of excitement to any occasion. With a heritage of quality and a commitment to innovation, every sip of FizzyPop delivers an unmatched experience. Embrace the bold, and make every moment extraordinary with the unmistakable taste of FizzyPop.",
    "Explore the skies with SkyTech, the world's leading aerospace company renowned for its innovation, quality, and reliability. SkyTech designs, manufactures, and services commercial airplanes, defense systems, and space technologies, making global connectivity and exploration possible. Whether you're traveling for business or leisure, SkyTech's state-of-the-art aircraft ensure a safe, comfortable, and efficient journey. With a legacy of pioneering advancements and a commitment to excellence, SkyTech continues to shape the future of aviation. Choose SkyTech and experience the pinnacle of aerospace engineering and performance. Fly with confidence, fly with SkyTech.",
    "Experience the future of aviation with AeroDynamics, the global leader in aerospace innovation and excellence. AeroDynamics designs and manufactures the world’s most advanced commercial aircraft, providing unparalleled comfort, efficiency, and reliability. From cutting-edge technology to sustainable solutions, AeroDynamics is dedicated to shaping the future of air travel. Whether you're embarking on a long-haul journey or a short domestic flight, AeroDynamics ensures a superior flying experience with spacious cabins, innovative features, and top-notch safety standards. Trust AeroDynamics for a seamless and enjoyable journey every time. Fly smarter, fly with AeroDynamics.",
    "Immerse yourself in the world of music with MusicStream, the ultimate destination for streaming your favorite tunes anytime, anywhere. With a vast library of millions of songs, playlists curated just for you, and personalized recommendations, MusicStream puts the power of music discovery in your hands. Whether you're in the mood for chart-topping hits, underground gems, or soothing melodies, MusicStream has something for everyone. Plus, with offline listening capabilities and seamless integration across devices, you can take your music with you wherever you go. Join the millions of music lovers worldwide and unlock endless possibilities with MusicStream. Discover, stream, and experience the joy of music like never before.", 
    "Experience the cutting-edge innovation of BrainChips, the global leader in semiconductor technology. BrainChips' groundbreaking processors power the devices that fuel our modern world, from laptops and desktops to servers and cloud computing systems. With a legacy of pushing the boundaries of technology, BrainChips continues to deliver industry-leading performance, reliability, and security. Whether you're a professional tackling complex tasks or a gamer seeking immersive experiences, BrainChips processors provide the power and efficiency you need. Trust BrainChips to deliver the performance you demand and the reliability you can count on. Join the millions who rely on BrainChips technology and unlock new possibilities for productivity, creativity, and entertainment.",
]

bids_dict = {advertiser: 1 for advertiser in ['Velora', 'BookHaven', 'MassMart', 'EspressoEdge', 'SocialHub', 'ColaBubbles', 'FizzyPop', 'SkyTech', 'AeroDynamics', 'MusicStream', 'BrainChips']}


In [28]:
q = rag_based_relevance(prompt, ads)
print([round(float(v), 2) for v in q])
for name, value in zip(advertisers, q):
    print(f'{name}: {value:.4f}')

[0.53, 0.7, 0.51, 0.49, 0.48, 0.52, 0.53, 0.5, 0.52, 0.52, 0.52]
Velora: 0.5251
BookHaven: 0.7048
MassMart: 0.5078
EspressoEdge: 0.4913
SocialHub: 0.4751
ColaBubbles: 0.5244
FizzyPop: 0.5338
SkyTech: 0.4994
AeroDynamics: 0.5156
MusicStream: 0.5191
BrainChips: 0.5164


In [29]:
def run_experiment_4(N, eta_val, beta_val, with_replacement, lambda_val=1.0):
    repl_str  = "with" if with_replacement else "without"
    save_path = f'QP_single_{repl_str}_replacement_logs_scenario_4.json'
    print(f"Running: N={N}, eta={eta_val}, beta={beta_val}, {repl_str}_replacement → {save_path}")
    run_N_trials(
        N                = N,
        bids_dict        = bids_dict,
        advertisers      = advertisers,
        ads              = ads,
        original_query   = prompt,
        eta_val          = eta_val,
        beta_val         = beta_val,
        lambda_val       = lambda_val,
        with_replacement = with_replacement,
        save_path        = save_path,
    )
    print(f"  Saved: {save_path}")
    return save_path

### with replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_4(N=20, eta_val=0.7, beta_val=0.8,with_replacement=True)
    except Exception as e:
        print(f"Error: {e}, retrying...")

### without replacement

In [ ]:
for _ in range(5):
    try:
        run_experiment_4(N=20, eta_val=0.7, beta_val=0.8,with_replacement=False)
    except Exception as e:
        print(f"Error: {e}, retrying...")